In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
events_path = RAW_DIR / "events.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", events_path.exists())

Project root: c:\YR_Python_Code\multi-stage-ads-recommendation\notebooks
Dataset exists: True


In [2]:

# Inspect all files in the raw data directory
files = sorted(RAW_DIR.iterdir())

if not files:
    print("No files found. Download and extract the dataset first.")

for file in files:
    if file.is_file():
        size_mb = file.stat().st_size / (1024 ** 2)
        print(f"{file.name}: {size_mb:.2f} MB")

# Check whether the main dataset exists
events_path = RAW_DIR / "events.csv"

assert events_path.exists(), (
    f"events.csv not found in {RAW_DIR}"
)

print("events.csv found successfully!")


category_tree.csv: 0.01 MB
events.csv: 89.87 MB
item_properties_part1.csv: 461.88 MB
item_properties_part2.csv: 389.99 MB
events.csv found successfully!


In [3]:

import pandas as pd
import numpy as np
import time

# Load the dataset
start_time = time.perf_counter()

events = pd.read_csv(events_path)

load_time = time.perf_counter() - start_time

print(f"Loading time: {load_time:.2f} seconds")
print(f"DataFrame shape: {events.shape}")

# Preview the first 5 rows
display(events.head())


Loading time: 0.81 seconds
DataFrame shape: (2756101, 5)


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [4]:

# Inspect schema
print("Column names:")
print(events.columns.tolist())

print("\nData types:")
print(events.dtypes)

print("\nDataFrame information:")
events.info()

# Validate expected columns
expected_columns = {
    "timestamp",
    "visitorid",
    "event",
    "itemid",
    "transactionid"
}

assert set(events.columns) == expected_columns, (
    "Unexpected dataset schema!"
)

print("\nSchema validation passed!")


Column names:
['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']

Data types:
timestamp          int64
visitorid          int64
event             object
itemid             int64
transactionid    float64
dtype: object

DataFrame information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          object 
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), object(1)
memory usage: 105.1+ MB

Schema validation passed!


In [5]:

# Total memory usage
memory_bytes = events.memory_usage(deep=True).sum()
memory_mb = memory_bytes / (1024 ** 2)

print(f"Total memory usage: {memory_mb:.2f} MiB")

# Memory usage by column
column_memory = (
    events.memory_usage(deep=True)
    / (1024 ** 2)
)

print("\nMemory usage by column (MiB):")
print(column_memory.sort_values(ascending=False))

# Bytes per row
bytes_per_row = memory_bytes / len(events)

print(f"\nAverage bytes per row: {bytes_per_row:.2f}")


Total memory usage: 244.92 MiB

Memory usage by column (MiB):
event            160.814304
timestamp         21.027382
itemid            21.027382
visitorid         21.027382
transactionid     21.027382
Index              0.000122
dtype: float64

Average bytes per row: 93.18


In [6]:

# Create a copy of the event column
event_original = events["event"]

# Convert string values to category
event_optimized = event_original.astype("category")

# Compare memory usage
original_mb = (
    event_original.memory_usage(deep=True)
    / (1024 ** 2)
)

optimized_mb = (
    event_optimized.memory_usage(deep=True)
    / (1024 ** 2)
)

reduction = (
    1 - optimized_mb / original_mb
) * 100

print(f"Original: {original_mb:.2f} MiB")
print(f"Optimized: {optimized_mb:.2f} MiB")
print(f"Memory reduction: {reduction:.2f}%")

print("\nCategories:")
print(event_optimized.cat.categories.tolist())

print("\nCategory codes:")
print(event_optimized.cat.codes.head())


Original: 160.81 MiB
Optimized: 2.63 MiB
Memory reduction: 98.37%

Categories:
['addtocart', 'transaction', 'view']

Category codes:
0    2
1    2
2    2
3    2
4    2
dtype: int8


In [7]:

# Count missing values in each column
missing_count = events.isna().sum()

# Calculate missing percentage
missing_percentage = (
    events.isna().mean() * 100
)

# Build a summary table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage (%)": missing_percentage
})

print("Missing Value Summary:")
display(missing_summary)

# Check total missing values
print(
    "Total missing cells:",
    events.isna().sum().sum()
)


Missing Value Summary:


,Missing Count,Missing Percentage (%)
timestamp,0,0.00000
visitorid,0,0.00000
event,0,0.00000
itemid,0,0.00000
transactionid,2733644,99.18519


Total missing cells: 2733644


In [8]:

# Check missing transaction IDs by event type
transaction_missing = (
    events.groupby("event")["transactionid"]
    .agg(
        total_events="size",
        missing_count=lambda x: x.isna().sum(),
        non_missing_count="count"
    )
)

transaction_missing["missing_pct"] = (
    transaction_missing["missing_count"]
    / transaction_missing["total_events"]
    * 100
)

display(transaction_missing)


,total_events,missing_count,non_missing_count,missing_pct
event,,,,
addtocart,69332,69332,0,100.0
transaction,22457,0,22457,0.0
view,2664312,2664312,0,100.0


In [9]:

# Count exact duplicate rows
duplicate_count = events.duplicated().sum()

duplicate_pct = (
    duplicate_count / len(events) * 100
)

print(f"Exact duplicates: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_pct:.4f}%")

# Inspect example duplicate records
duplicate_examples = events[
    events.duplicated(keep=False)
]

display(duplicate_examples.head(10))


Exact duplicates: 460
Duplicate percentage: 0.0167%


,timestamp,visitorid,event,itemid,transactionid
13734,1433180781440,1045411,addtocart,379647,NaN
22404,1433180781440,1045411,addtocart,379647,NaN
24476,1433274223925,366538,addtocart,252068,NaN
25508,1433265100661,198153,addtocart,48715,NaN
27095,1433270868154,1268755,view,60980,NaN
34465,1433274223925,366538,addtocart,252068,NaN
34578,1433277134682,555487,addtocart,397642,NaN
39738,1433265100661,198153,addtocart,48715,NaN
41774,1433270868154,1268755,view,60980,NaN
42469,1433277134682,555487,addtocart,397642,NaN


In [ ]:

# Check unique event types
print("Event types:")
print(events["event"].value_counts(dropna=False))

# Validate expected events
valid_events = {"view", "addtocart", "transaction"}

invalid_event_mask = (
    ~events["event"].isin(valid_events)
)

print(
    "\nInvalid event records:",
    invalid_event_mask.sum()
)

# Check invalid identifiers and timestamps
for col in ["visitorid", "itemid", "timestamp"]:
    invalid_count = (
        events[col].isna() | (events[col] < 0)
    ).sum()

    print(f"Invalid {col}: {invalid_count:,}")

# Timestamp should be positive
assert (events["timestamp"].dropna() > 0).all()


In [ ]:

# Inspect raw timestamps
print("Timestamp dtype:", events["timestamp"].dtype)

print("\nFirst 5 raw timestamps:")
print(events["timestamp"].head())

# Convert Unix milliseconds to UTC datetime
events["datetime"] = pd.to_datetime(
    events["timestamp"],
    unit="ms",
    utc=True,
    errors="coerce"
)

print("\nFirst 5 converted timestamps:")
display(events[["timestamp", "datetime"]].head())

print("\nDatetime dtype:")
print(events["datetime"].dtype)

# Check failed conversions
print(
    "\nInvalid datetime count:",
    events["datetime"].isna().sum()
)


In [ ]:

# Find earliest and latest timestamps
min_time = events["datetime"].min()
max_time = events["datetime"].max()

# Calculate dataset duration
duration = max_time - min_time

print("Start time:", min_time)
print("End time:", max_time)
print("Duration:", duration)
print("Duration in days:", duration.total_seconds() / 86400)

# Count unique calendar dates
unique_days = events["datetime"].dt.date.nunique()

print("Unique active days:", unique_days)


In [ ]:

import matplotlib.pyplot as plt

# Count events per UTC day
daily_events = (
    events.set_index("datetime")
    .resample("D")
    .size()
)

print("Daily event statistics:")
print(daily_events.describe())

# Plot daily event counts
plt.figure(figsize=(12, 5))

plt.plot(
    daily_events.index,
    daily_events.values
)

plt.title("Daily Interaction Volume")
plt.xlabel("Date (UTC)")
plt.ylabel("Number of Events")

plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Count events by day and event type
daily_event_types = (
    events.groupby([
        pd.Grouper(key="datetime", freq="D"),
        "event"
    ])
    .size()
    .unstack(fill_value=0)
)

display(daily_event_types.head())

# Plot each event type separately
fig, axes = plt.subplots(
    3, 1,
    figsize=(12, 9),
    sharex=True
)

for ax, event_type in zip(
    axes,
    ["view", "addtocart", "transaction"]
):
    ax.plot(
        daily_event_types.index,
        daily_event_types[event_type]
    )

    ax.set_title(f"Daily {event_type} Events")
    ax.set_ylabel("Count")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Date (UTC)")
plt.tight_layout()
plt.show()


In [ ]:

# Check whether events are chronologically sorted
is_sorted = events["timestamp"].is_monotonic_increasing

print("Is dataset sorted by timestamp?", is_sorted)

# Check timestamp uniqueness
unique_timestamps = events["timestamp"].nunique()

print("Unique timestamps:", unique_timestamps)

# Count repeated timestamps
print(
    "Rows sharing a timestamp with another row:",
    events["timestamp"].duplicated(keep=False).sum()
)


In [ ]:

# Basic dataset statistics

n_events = len(events)

n_users = events["visitorid"].nunique()
n_items = events["itemid"].nunique()

print(f"Total events: {n_events:,}")
print(f"Unique users: {n_users:,}")
print(f"Unique items: {n_items:,}")

# Average events per user
avg_events_per_user = n_events / n_users

# Average events per item
avg_events_per_item = n_events / n_items

print(f"\nAverage events per user: {avg_events_per_user:.2f}")
print(f"Average events per item: {avg_events_per_item:.2f}")


In [ ]:

# Count interactions per user
user_activity = events["visitorid"].value_counts()

print("User activity statistics:")
print(user_activity.describe(
    percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
))

print("\nTop 10 most active users:")
display(user_activity.head(10))

# Count users with only one interaction
single_event_users = (user_activity == 1).sum()

single_event_pct = (
    single_event_users / n_users * 100
)

print(f"\nUsers with one event: {single_event_users:,}")
print(f"Percentage: {single_event_pct:.2f}%")


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10, 5))

plt.hist(
    user_activity,
    bins=100
)

plt.xlabel("Number of Events per User")
plt.ylabel("Number of Users")
plt.title("User Activity Distribution")
plt.yscale("log")
plt.grid(alpha=0.3)
plt.show()


In [ ]:

# Count events per item
item_popularity = events["itemid"].value_counts()

print("Item popularity statistics:")
print(item_popularity.describe(
    percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
))

print("\nTop 10 most popular items:")
display(item_popularity.head(10))

# Items with only one interaction
single_event_items = (item_popularity == 1).sum()

single_item_pct = (
    single_event_items / n_items * 100
)

print(f"\nItems with one event: {single_event_items:,}")
print(f"Percentage: {single_item_pct:.2f}%")


In [ ]:

# Sort items from most to least popular
sorted_popularity = item_popularity.sort_values(
    ascending=False
).to_numpy()

# Item rank: 1, 2, 3, ...
item_rank = np.arange(
    1, len(sorted_popularity) + 1
)

plt.figure(figsize=(10, 5))

plt.loglog(
    item_rank,
    sorted_popularity
)

plt.xlabel("Item Popularity Rank (log)")
plt.ylabel("Number of Interactions (log)")
plt.title("Item Popularity Rank-Frequency Curve")
plt.grid(alpha=0.3)
plt.show()


In [ ]:

# Sort item popularity in descending order
sorted_counts = item_popularity.sort_values(
    ascending=False
)

# Top 1% and 10% of observed items
top_1_n = max(1, int(np.ceil(n_items * 0.01)))
top_10_n = max(1, int(np.ceil(n_items * 0.10)))

top_1_share = (
    sorted_counts.head(top_1_n).sum() / n_events
)

top_10_share = (
    sorted_counts.head(top_10_n).sum() / n_events
)

print(f"Top 1% items: {top_1_share:.2%} of events")
print(f"Top 10% items: {top_10_share:.2%} of events")


In [ ]:

# Count unique user-item pairs
unique_pairs = (
    events[["visitorid", "itemid"]]
    .drop_duplicates()
    .shape[0]
)

# Total possible pairs
possible_pairs = int(n_users) * int(n_items)

# Observed interaction density
density = unique_pairs / possible_pairs

sparsity = 1 - density

print(f"Unique user-item pairs: {unique_pairs:,}")
print(f"Possible user-item pairs: {possible_pairs:,}")
print(f"Observed density: {density:.6%}")
print(f"Sparsity: {sparsity:.6%}")


In [ ]:

# Group users based on interaction counts

user_segments = {
    "1 event": (user_activity == 1).sum(),
    "2-5 events": (
        (user_activity >= 2) &
        (user_activity <= 5)
    ).sum(),
    "6-20 events": (
        (user_activity >= 6) &
        (user_activity <= 20)
    ).sum(),
    ">20 events": (user_activity > 20).sum()
}

user_segment_df = pd.DataFrame({
    "segment": list(user_segments.keys()),
    "user_count": list(user_segments.values())
})

user_segment_df["percentage"] = (
    user_segment_df["user_count"] / n_users * 100
)

display(user_segment_df)

assert user_segment_df["user_count"].sum() == n_users


In [ ]:

# Group items based on interaction counts

item_segments = {
    "1 event": (item_popularity == 1).sum(),
    "2-5 events": (
        (item_popularity >= 2) &
        (item_popularity <= 5)
    ).sum(),
    "6-20 events": (
        (item_popularity >= 6) &
        (item_popularity <= 20)
    ).sum(),
    ">20 events": (item_popularity > 20).sum()
}

item_segment_df = pd.DataFrame({
    "segment": list(item_segments.keys()),
    "item_count": list(item_segments.values())
})

item_segment_df["percentage"] = (
    item_segment_df["item_count"] / n_items * 100
)

display(item_segment_df)

assert item_segment_df["item_count"].sum() == n_items


In [ ]:

# Map total activity counts to each event
events["user_event_count"] = (
    events["visitorid"].map(user_activity)
)

# Define activity segments
def assign_user_segment(count):
    if count == 1:
        return "1 event"
    elif count <= 5:
        return "2-5 events"
    elif count <= 20:
        return "6-20 events"
    else:
        return ">20 events"

events["user_segment"] = (
    events["user_event_count"].apply(assign_user_segment)
)

# Count event types by segment
segment_event_counts = pd.crosstab(
    events["user_segment"],
    events["event"]
)

print("Event counts:")
display(segment_event_counts)

# Normalize each row to percentages
segment_event_pct = (
    segment_event_counts.div(
        segment_event_counts.sum(axis=1),
        axis=0
    ) * 100
)

print("Event percentages:")
display(segment_event_pct.round(2))


In [ ]:

# Validate basic dataset properties

assert len(events) == 2_756_101

assert events["visitorid"].notna().all()
assert events["itemid"].notna().all()
assert events["timestamp"].notna().all()

assert events["event"].isin([
    "view", "addtocart", "transaction"
]).all()

assert events["datetime"].notna().all()

# Transaction events should have transaction IDs
transaction_mask = events["event"] == "transaction"

assert events.loc[
    transaction_mask, "transactionid"
].notna().all()

# Non-transaction events should have missing transaction IDs
assert events.loc[
    ~transaction_mask, "transactionid"
].isna().all()

print("All basic data validation checks passed!")


Total events
2,756,101

Unique users
1,407,580

Unique items
235,061

User-item sparsity
99.999352%



Metric	Result
Dataset period	                May 3 – Sep 18, 2015
Unique active UTC dates	139
View events	2,664,312
Add-to-cart events	69,332
Transaction events	22,457
Exact duplicates	460 (0.0167%)
Invalid timestamps	0
Invalid IDs/event types	0
Transaction ID missingness	Consistent with event types
Basic validation	Passed